# Tokamak Hackathon — предсказание срыва плазмы

Ноутбук сравнивает три варианта признаков на **одинаковом разбиении по разрядам**:

1. **Raw** — измеряемые каналы и время, без ручной инженерии признаков.
2. **Physics** — исходные каналы плюс интерпретируемые физические прокси.
3. **Dynamics** — физические признаки плюс лаги, разности и скользящие статистики, использующие только текущие и прошлые измерения.

Цель — предсказать, произойдёт ли термическое гашение в ближайшие `LEAD_TIME_S` секунд. Используется `t_tq_s` из `shots.csv`; `t_trigger_s` не подменяет время термического гашения.

**Принципиальные ограничения:** `shot_id` нужен для группировки, но не является признаком; поля из `shots.csv` не передаются модели; после `t_tq_s` строки зривного разряда удаляются; в лаговых признаках запрещено заполнять пропуски будущими значениями (`bfill`). Метрики предупреждения считаются по разрядам, а не только по строкам.

Сначала запусти ячейки сверху вниз. Файлы `measured.csv` и `shots.csv` должны лежать рядом с ноутбуком.

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# XGBoost обычно хорошо подходит для табличных данных.
# Если он не установлен, используем sklearn HistGradientBoostingClassifier.
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
    MODEL_BACKEND = "XGBoost"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGBOOST = False
    MODEL_BACKEND = "HistGradientBoostingClassifier"

RANDOM_STATE = 42
MEASURED_FILE = Path("measured.csv")
SHOTS_FILE = Path("shots.csv")

# Горизонт целевой метки (секунды до t_Q).
# Для сравнения вариантов используем один и тот же горизонт.
LEAD_TIME_S = 0.30

# Допустимая доля безопасных разрядов, на которых появится ложная тревога.
FALSE_ALARM_BUDGET = 0.05
VALID_SIZE = 0.25
N_JOBS = 4

print("Backend:", MODEL_BACKEND)
print("Lead time:", LEAD_TIME_S, "s")
print("False-alarm budget:", f"{FALSE_ALARM_BUDGET:.1%}")

Backend: XGBoost
Lead time: 0.3 s
False-alarm budget: 5.0%


In [2]:
# 1. Загрузка и первичная проверка данных
if not MEASURED_FILE.exists():
    raise FileNotFoundError(f"Не найден файл: {MEASURED_FILE.resolve()}")
if not SHOTS_FILE.exists():
    raise FileNotFoundError(f"Не найден файл: {SHOTS_FILE.resolve()}")

measured = pd.read_csv(MEASURED_FILE)
shots = pd.read_csv(SHOTS_FILE)

# Иногда CSV после экспорта содержит лишний индексный столбец.
measured = measured.loc[:, ~measured.columns.astype(str).str.startswith("Unnamed:")].copy()
shots = shots.loc[:, ~shots.columns.astype(str).str.startswith("Unnamed:")].copy()

required_measured = {"shot_id", "t", "mag_ip_MA", "mag_bt_T"}
required_shots = {"shot_id", "disrupted", "t_tq_s"}
missing_m = required_measured - set(measured.columns)
missing_s = required_shots - set(shots.columns)
if missing_m:
    raise ValueError(f"В measured.csv не хватает обязательных колонок: {sorted(missing_m)}")
if missing_s:
    raise ValueError(
        f"В shots.csv не хватает колонок: {sorted(missing_s)}. Нужен именно t_tq_s — время термического гашения."
    )

# Типы shot_id должны совпадать в обоих файлах.
measured["shot_id"] = pd.to_numeric(measured["shot_id"], errors="raise").astype("int64")
shots["shot_id"] = pd.to_numeric(shots["shot_id"], errors="raise").astype("int64")
measured["t"] = pd.to_numeric(measured["t"], errors="coerce")
shots["t_tq_s"] = pd.to_numeric(shots["t_tq_s"], errors="coerce")

if shots["shot_id"].duplicated().any():
    raise ValueError("В shots.csv есть повторяющиеся shot_id: ожидается одна строка на разряд.")

# Все колонки measured, кроме shot_id, должны быть числовыми измерениями.
# Пустые ячейки остаются NaN: это отказ/пропуск канала, а не нулевое измерение.
measured_channels = [c for c in measured.columns if c != "shot_id"]
for col in measured_channels:
    measured[col] = pd.to_numeric(measured[col], errors="coerce")

print(f"measured: {measured.shape[0]:,} строк × {measured.shape[1]} колонок")
print(f"shots:    {shots.shape[0]:,} разрядов × {shots.shape[1]} колонок")
print("Измеряемых признаков (без shot_id):", len(measured_channels))
print("Временной шаг, медиана по разрядам:")
print(measured.sort_values(["shot_id", "t"]).groupby("shot_id")["t"].diff().median())

display(shots.head())
display(measured.head(3))

measured: 272,176 строк × 102 колонок
shots:    100 разрядов × 9 колонок
Измеряемых признаков (без shot_id): 101
Временной шаг, медиана по разрядам:
0.004999999999999893


,shot_id,scenario,disrupted,t_trigger_s,t_tq_s,spontaneous,wall_contact,n_steps,t_end_s
0,0,density+heat_loss,False,NaN,NaN,False,False,2990,14.950
1,1,low_q,False,NaN,NaN,False,False,3196,15.980
2,2,low_q+density,True,11.685,11.775,False,False,2354,11.770
3,3,density,True,11.015,11.080,True,False,2215,11.075
4,4,none,False,NaN,NaN,False,False,2801,14.005


,shot_id,t,mag_ip_MA,mag_bt_T,mag_diamag_MJ,mag_vloop_V,mag_mirnov_01,mag_mirnov_02,mag_mirnov_03,mag_mirnov_04,...,nbi_beam_energy_keV,nbi_source_current_A,nbi_source_2_current_A,ich_antenna_voltage_kV,prog_ip_MA,prog_bt_T,prog_ne_1e20,prog_p_nbi_MW,prog_p_ech_MW,prog_p_ich_MW
0,0,0.005,0.004,3.3236,0.00000,0.008,-0.001500,0.000750,-0.000375,-0.001125,...,107.05,0.0,0.0,0.0,0.003813,3.32099,0.028626,0.0,0.0,0.0
1,0,0.010,0.008,3.3271,0.00225,-0.008,-0.002625,-0.001875,-0.001875,-0.000375,...,100.25,0.0,0.0,0.0,0.007626,3.32099,0.029339,0.0,0.0,0.0
2,0,0.015,0.012,3.3250,0.00000,0.012,-0.002625,0.001125,-0.001125,0.001500,...,101.15,0.0,0.0,0.0,0.011439,3.32099,0.030052,0.0,0.0,0.0


## 2. Подготовка таргета

Для зривного разряда метка равна 1, если `0 < t_tq_s - t <= LEAD_TIME_S`; раньше этого окна и для безопасных разрядов метка равна 0. Строки при `t >= t_tq_s` исключаются, поскольку это уже не прогноз до гашения.

Разряды со срывом, помеченным как `spontaneous` или `wall_contact`, а также зривные разряды без `t_tq_s`, исключаются из обучения и оценки: методичка относит их к `Excluded`. Безопасные разряды сохраняются.

In [3]:
def as_bool(series: pd.Series) -> pd.Series:
    """Надёжно переводит CSV-значения true/false в bool."""
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y", "t"})

shots = shots.copy()
shots["disrupted_bool"] = as_bool(shots["disrupted"])
if "spontaneous" in shots:
    shots["spontaneous_bool"] = as_bool(shots["spontaneous"])
else:
    shots["spontaneous_bool"] = False
if "wall_contact" in shots:
    shots["wall_contact_bool"] = as_bool(shots["wall_contact"])
else:
    shots["wall_contact_bool"] = False

bad_disruption = (
    shots["disrupted_bool"]
    & (
        shots["spontaneous_bool"]
        | shots["wall_contact_bool"]
        | shots["t_tq_s"].isna()
    )
)
excluded_shot_ids = set(shots.loc[bad_disruption, "shot_id"])
shots_eligible = shots.loc[~bad_disruption].copy()

# Объединяем только служебные поля для построения метки/оценки.
# Эти поля не войдут в матрицу признаков.
meta_cols = [
    "shot_id", "disrupted_bool", "t_tq_s",
    "spontaneous_bool", "wall_contact_bool",
]
df = measured.merge(
    shots_eligible[meta_cols],
    on="shot_id",
    how="inner",
    validate="many_to_one",
)
df = df.sort_values(["shot_id", "t"]).reset_index(drop=True)

# После t_Q нет прогноза: убираем строки гашения и возможные строки после него.
post_quench = df["disrupted_bool"] & (df["t"] >= df["t_tq_s"])
df = df.loc[~post_quench].copy().reset_index(drop=True)

time_to_tq = df["t_tq_s"] - df["t"]
df["target_y"] = (
    df["disrupted_bool"]
    & (time_to_tq > 0)
    & (time_to_tq <= LEAD_TIME_S)
).astype("int8")

# Проверка качества метки и числа разрядов.
y = df["target_y"].astype("int8")
shot_info = shots_eligible[meta_cols].drop_duplicates("shot_id").copy()
print(f"Исключено непригодных зривных разрядов: {len(excluded_shot_ids)}")
print(f"Оставлено разрядов: {shot_info['shot_id'].nunique()}")
print(f"Строк после удаления данных t >= t_Q: {len(df):,}")
print(f"Положительных строк: {int(y.sum()):,} ({y.mean():.4%})")
print("Исходы разрядов:")
display(shot_info["disrupted_bool"].value_counts().rename(index={False: "без срыва", True: "допустимый срыв"}).to_frame("shots"))

if y.sum() == 0:
    raise ValueError(
        "Таргет не содержит положительных строк. Проверь t_tq_s и LEAD_TIME_S; не заменяй t_tq_s на t_trigger_s автоматически."
    )
if shot_info["disrupted_bool"].nunique() < 2:
    raise ValueError("Нужны как безопасные, так и зривные разряды для сравнения модели.")

Исключено непригодных зривных разрядов: 7
Оставлено разрядов: 93
Строк после удаления данных t >= t_Q: 260,953
Положительных строк: 2,078 (0.7963%)
Исходы разрядов:


,shots
disrupted_bool,
без срыва,58
допустимый срыв,35


## 3. Три набора признаков

- **Raw:** только исходные измеряемые каналы и время `t`.
- **Physics:** исходные каналы плюс агрегаты диагностик и приближённые физические соотношения. Это именно *прокси*, а не восстановленные истинные `q95`, `f_GW` или `P_rad`: в предоставленном `measured.csv` нет их прямых откалиброванных колонок.
- **Dynamics:** Physics плюс лаги и скользящие статистики. Окна считаются по предыдущим строкам и текущей строке внутри того же разряда. `bfill` нигде не используется.

Все исходные каналы из `measured.csv` оставляем как есть: в частности, управляющие/программные каналы не берутся из `shots.csv`, а подаются только если действительно присутствуют среди входных измерений.

In [4]:
def _row_agg(frame: pd.DataFrame, columns, how="mean") -> pd.Series:
    cols = [c for c in columns if c in frame.columns]
    if not cols:
        return pd.Series(np.nan, index=frame.index, dtype=float)
    values = frame[cols]
    if how == "mean":
        return values.mean(axis=1)
    if how == "max":
        return values.max(axis=1)
    if how == "min":
        return values.min(axis=1)
    if how == "std":
        return values.std(axis=1)
    if how == "rms":
        return np.sqrt((values ** 2).mean(axis=1))
    if how == "abs_sum":
        return values.abs().sum(axis=1, min_count=1)
    raise ValueError(f"Неизвестная агрегация: {how}")


def make_physics_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Исходные измерения + читаемые физические/диагностические прокси."""
    out = raw.copy()
    eps_ip = 0.03  # стабилизация отношений в самом начале разряда, когда Ip ~ 0

    # Нагрев: реальные измеряемые мощности, если эти каналы присутствуют.
    heating_cols = ["nbi_power_MW", "ich_power_MW", "ech_power_MW"]
    present_heating = [c for c in heating_cols if c in out.columns]
    out["feat_input_power_MW"] = (
        out[present_heating].sum(axis=1, min_count=1) if present_heating
        else np.nan
    )

    # База геометрического масштабирования q95 без shape factor f_s.
    # Не выдаём этот прокси за истинное q95: измеренные kappa/delta в данном списке нет.
    if {"mag_bt_T", "mag_ip_MA"}.issubset(out.columns):
        out["feat_q95_base_proxy"] = (
            5.0 * (0.95 ** 2) * out["mag_bt_T"].abs()
            / (2.89 * (out["mag_ip_MA"].abs() + eps_ip))
        )
        out["feat_ip_over_bt"] = out["mag_ip_MA"].abs() / (out["mag_bt_T"].abs() + 0.05)

    # Среднее/разброс хорд — только прокси плотности; без калибровки это не f_GW.
    chord_cols = [c for c in out.columns if c.startswith("ifr_chord_")]
    if chord_cols:
        out["feat_chord_mean"] = _row_agg(out, chord_cols, "mean")
        out["feat_chord_spread"] = (
            _row_agg(out, chord_cols, "max") - _row_agg(out, chord_cols, "min")
        )
        if "mag_ip_MA" in out.columns:
            out["feat_chord_per_current_proxy"] = (
                out["feat_chord_mean"] / (out["mag_ip_MA"].abs() + eps_ip)
            )

    # Флуктуации магнитного поля: RMS по Mirnov-каналам.
    mirnov_cols = [c for c in out.columns if c.startswith("mag_mirnov_")]
    if mirnov_cols:
        out["feat_mirnov_rms"] = _row_agg(out, mirnov_cols, "rms")

    # D-alpha — индикатор активности на краю/ELM. Используем абсолютные амплитуды,
    # так как у синтетических каналов возможен небольшой шум вокруг нуля.
    dalpha_cols = [c for c in out.columns if c.startswith("dalpha_")]
    if dalpha_cols:
        out["feat_dalpha_mean_abs"] = out[dalpha_cols].abs().mean(axis=1)
        out["feat_dalpha_max_abs"] = out[dalpha_cols].abs().max(axis=1)

    # ECE и болометрия агрегируются как диагностические сигналы, а не как абсолютные MW.
    ece_cols = [c for c in out.columns if c.startswith("ece_")]
    if ece_cols:
        out["feat_ece_mean"] = _row_agg(out, ece_cols, "mean")
        out["feat_ece_spread"] = _row_agg(out, ece_cols, "max") - _row_agg(out, ece_cols, "min")

    bolo_cols = [c for c in out.columns if c.startswith("bolo_")]
    if bolo_cols:
        out["feat_bolo_abs_sum"] = _row_agg(out, bolo_cols, "abs_sum")
        out["feat_bolo_mean"] = _row_agg(out, bolo_cols, "mean")

    # P_input / W_diamag — приближённая частота нагрева относительно запасённой энергии.
    # Она не равна полной обратной tau_E: потери и dW/dt здесь не восстановлены.
    if {"feat_input_power_MW", "mag_diamag_MJ"}.issubset(out.columns):
        out["feat_heating_to_energy_s_inv_proxy"] = (
            out["feat_input_power_MW"] / (out["mag_diamag_MJ"].abs() + 0.02)
        )

    # V_loop / I_p — прокси электрического сопротивления плазмы.
    if {"mag_vloop_V", "mag_ip_MA"}.issubset(out.columns):
        out["feat_vloop_per_current_proxy"] = (
            out["mag_vloop_V"].abs() / (out["mag_ip_MA"].abs() + eps_ip)
        )

    if "mag_saddle_lm" in out.columns:
        out["feat_locked_mode_abs"] = out["mag_saddle_lm"].abs()

    if "neutron_total" in out.columns:
        out["feat_neutron_log1p"] = np.log1p(out["neutron_total"].clip(lower=0))

    if {"gas_flow_ne_meas", "gas_flow_d2_meas"}.issubset(out.columns):
        out["feat_neon_to_d2_flow_proxy"] = (
            out["gas_flow_ne_meas"].abs() / (out["gas_flow_d2_meas"].abs() + 1e-8)
        )

    return out.replace([np.inf, -np.inf], np.nan)


def make_dynamic_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Physics-признаки + строго причинные лаги/тренды внутри каждого shot_id."""
    out = make_physics_features(raw)
    if "shot_id" not in out.columns:
        raise ValueError("Для динамических признаков нужен shot_id, чтобы не смешивать разряды.")

    # В измерениях из примера dt=5 ms: 1/3/10/20 шагов ~= 5/15/50/100 ms.
    # Если Dataset экспортирован с Sample every > 1, интервалы растут пропорционально.
    lag_sources = [
        "mag_saddle_lm", "mag_diamag_MJ", "mag_vloop_V", "ifr_chord_core",
        "ece_08", "neutron_total", "feat_dalpha_max_abs", "feat_mirnov_rms",
        "feat_input_power_MW",
    ]
    lag_sources = [c for c in lag_sources if c in out.columns]
    for col in lag_sources:
        grouped = out.groupby("shot_id", sort=False)[col]
        for lag in (1, 3, 10, 20):
            out[f"dyn__{col}__lag{lag}"] = grouped.shift(lag)

    delta_sources = [
        "mag_saddle_lm", "mag_diamag_MJ", "ifr_chord_core", "ece_08",
        "neutron_total", "feat_dalpha_max_abs", "feat_mirnov_rms",
    ]
    for col in [c for c in delta_sources if c in out.columns]:
        grouped = out.groupby("shot_id", sort=False)[col]
        for lag in (10, 50):
            out[f"dyn__{col}__delta{lag}"] = out[col] - grouped.shift(lag)

    rolling_sources = [
        "mag_saddle_lm", "mag_diamag_MJ", "ifr_chord_core", "ece_08",
        "neutron_total", "feat_dalpha_max_abs", "feat_mirnov_rms",
    ]
    for col in [c for c in rolling_sources if c in out.columns]:
        grouped = out.groupby("shot_id", sort=False)[col]
        for window in (10, 50):
            out[f"dyn__{col}__mean{window}"] = grouped.transform(
                lambda s, w=window: s.rolling(window=w, min_periods=1).mean()
            )
            out[f"dyn__{col}__std{window}"] = grouped.transform(
                lambda s, w=window: s.rolling(window=w, min_periods=2).std()
            )

    return out.replace([np.inf, -np.inf], np.nan)


def model_matrix(feature_frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет идентификатор разряда, оставляет только числовые входные признаки."""
    X = feature_frame.drop(columns=["shot_id"], errors="ignore").copy()
    X = X.select_dtypes(include=[np.number])
    return X.replace([np.inf, -np.inf], np.nan)

# Только столбцы measured.csv используются для построения X.
raw_inputs = df[measured.columns].copy()
raw_inputs = raw_inputs.sort_values(["shot_id", "t"]).reset_index(drop=True)

feature_sets = {
    "Raw": model_matrix(raw_inputs),
    "Physics": model_matrix(make_physics_features(raw_inputs)),
    "Dynamics": model_matrix(make_dynamic_features(raw_inputs)),
}

# У всех вариантов тот же порядок строк, что и у таргета.
y = df["target_y"].astype("int8").reset_index(drop=True)
shot_ids = df["shot_id"].reset_index(drop=True)
for name, X in feature_sets.items():
    if len(X) != len(y):
        raise AssertionError(f"Несовпадение строк для {name}: X={len(X)}, y={len(y)}")
    print(f"{name:10s}: {X.shape[0]:,} строк × {X.shape[1]} признаков")

Raw       : 260,953 строк × 101 признаков
Physics   : 260,953 строк × 119 признаков
Dynamics  : 260,953 строк × 197 признаков


## 4. Разбиение по разрядам и обучение

Разделяем **список разрядов**, а не временные строки: все строки одного `shot_id` целиком принадлежат либо train, либо validation. Используем один и тот же разрез и один тип классификатора для всех трёх наборов признаков.

Из-за редких положительных строк немного усиливаем их вес. Это помогает обучению, но сырые выходы модели не следует считать идеально откалиброванными вероятностями; для хакатона дополнительно выбирается рабочая точка по ложным тревогам на разрядах.

In [5]:
# Стратифицируем на уровне разряда по факту допустимого срыва.
shot_summary = shot_info[["shot_id", "disrupted_bool"]].drop_duplicates("shot_id").copy()
class_counts = shot_summary["disrupted_bool"].value_counts()
if len(class_counts) < 2 or class_counts.min() < 2:
    raise ValueError(
        "Для группового holdout нужно минимум по два разряда каждого исхода. "
        "Сгенерируй больше разрядов в Dataset и повтори запуск."
    )

train_ids, valid_ids = train_test_split(
    shot_summary["shot_id"].to_numpy(),
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=shot_summary["disrupted_bool"].astype(int).to_numpy(),
)
train_ids, valid_ids = set(train_ids), set(valid_ids)
train_mask = shot_ids.isin(train_ids).to_numpy()
valid_mask = shot_ids.isin(valid_ids).to_numpy()

print(f"Train: {len(train_ids)} разрядов, {int(train_mask.sum()):,} строк")
print(f"Validation: {len(valid_ids)} разрядов, {int(valid_mask.sum()):,} строк")
print("Срывы в train/validation:",
      int(shot_summary[shot_summary.shot_id.isin(train_ids)].disrupted_bool.sum()), "/",
      int(shot_summary[shot_summary.shot_id.isin(valid_ids)].disrupted_bool.sum()))


def positive_class_weight(y_train: pd.Series) -> float:
    counts = np.bincount(np.asarray(y_train, dtype=int), minlength=2)
    if counts[1] == 0:
        raise ValueError(
            "В train нет положительных строк. Нужен больший датасет/другая групповая выборка."
        )
    # Умеренное усиление редкого класса, чтобы не перегнуть к ложным тревогам.
    return float(np.clip(np.sqrt(counts[0] / counts[1]), 1.0, 20.0))


def make_classifier(pos_weight: float):
    if HAS_XGBOOST:
        return XGBClassifier(
            n_estimators=350,
            max_depth=4,
            learning_rate=0.035,
            min_child_weight=5,
            subsample=0.85,
            colsample_bytree=0.80,
            gamma=0.10,
            reg_alpha=0.2,
            reg_lambda=5.0,
            scale_pos_weight=pos_weight,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        )
    return HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.06,
        max_leaf_nodes=15,
        l2_regularization=2.0,
        random_state=RANDOM_STATE,
    )


def fit_classifier(X_train: pd.DataFrame, y_train: pd.Series):
    weight = positive_class_weight(y_train)
    model = make_classifier(weight)
    if HAS_XGBOOST:
        model.fit(X_train, y_train)
    else:
        sample_weight = np.where(np.asarray(y_train) == 1, weight, 1.0)
        model.fit(X_train, y_train, sample_weight=sample_weight)
    return model


def get_probabilities(model, X: pd.DataFrame) -> np.ndarray:
    p = np.asarray(model.predict_proba(X)[:, 1], dtype=float)
    return np.clip(np.nan_to_num(p, nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0)


def row_ranking_metrics(y_true, p) -> dict:
    y_true = np.asarray(y_true, dtype=int)
    if len(np.unique(y_true)) < 2:
        return {"roc_auc": np.nan, "pr_auc": np.nan}
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
    }

Train: 69 разрядов, 194,718 строк
Validation: 24 разрядов, 66,235 строк
Срывы в train/validation: 26 / 9


## 5. Оценка тревоги на уровне разряда

Для каждого разряда тревога возникает при первом `p >= threshold`. `Caught` означает, что первая тревога появилась минимум за `LEAD_TIME_S` до `t_tq_s`. На безопасном разряде любая тревога — ложная.

Порог подбирается на validation так, чтобы доля ложных тревог не превысила `FALSE_ALARM_BUDGET`, а доля `Caught` была максимальной. Это **выбор рабочей точки**, поэтому его результат оптимистичен на этой же validation-выборке. Для финального обоснования повтори проверку на другом `Master seed` и через настоящий `toksym Test run`.

In [6]:
def shot_level_metrics(scored_rows: pd.DataFrame, validation_shots: pd.DataFrame,
                       threshold: float, lead_time_s: float) -> dict:
    alarms = (
        scored_rows.loc[scored_rows["p"] >= threshold]
        .groupby("shot_id")["t"].min()
    )
    safe_shots = validation_shots.loc[~validation_shots["disrupted_bool"]]
    event_shots = validation_shots.loc[validation_shots["disrupted_bool"]]

    false_alarms = int(sum(sid in alarms.index for sid in safe_shots["shot_id"]))
    n_safe = int(len(safe_shots))

    caught = 0
    late = 0
    missed = 0
    for row in event_shots.itertuples(index=False):
        alarm_t = alarms.get(row.shot_id, np.nan)
        if pd.isna(alarm_t):
            missed += 1
        elif (row.t_tq_s - alarm_t) >= lead_time_s:
            caught += 1
        else:
            late += 1

    n_events = int(len(event_shots))
    return {
        "threshold": float(threshold),
        "caught": caught,
        "n_events": n_events,
        "caught_rate": caught / n_events if n_events else np.nan,
        "late": late,
        "missed": missed,
        "false_alarms": false_alarms,
        "n_safe": n_safe,
        "false_alarm_rate": false_alarms / n_safe if n_safe else np.nan,
    }


def choose_threshold(scored_rows: pd.DataFrame, validation_shots: pd.DataFrame,
                     lead_time_s: float, false_alarm_budget: float):
    """Подбирает порог по максимумам безопасных разрядов и ранним максимумам срывных."""
    per_shot_max = scored_rows.groupby("shot_id")["p"].max()
    safe_ids = set(validation_shots.loc[~validation_shots["disrupted_bool"], "shot_id"])
    safe_max = per_shot_max.reindex(list(safe_ids)).dropna().to_numpy()
    if len(safe_max) == 0:
        raise ValueError("В validation нет безопасных разрядов — нельзя выставить бюджет ложных тревог.")

    # Для Caught важен максимум только за t <= t_Q - Lead time.
    early_rows = scored_rows.loc[
        scored_rows["disrupted_bool"]
        & scored_rows["t_tq_s"].notna()
        & (scored_rows["t"] <= scored_rows["t_tq_s"] - lead_time_s)
    ]
    early_max = early_rows.groupby("shot_id")["p"].max().dropna().to_numpy()

    raw_candidates = np.concatenate([safe_max, early_max])
    candidates = np.unique(np.concatenate([
        raw_candidates,
        np.nextafter(raw_candidates, np.inf),
    ]))
    candidates = candidates[np.isfinite(candidates)]
    candidates = np.clip(candidates, 0.0, 1.0)

    validation_shots = validation_shots.copy()
    options = []
    for threshold in candidates:
        m = shot_level_metrics(scored_rows, validation_shots, float(threshold), lead_time_s)
        if np.isfinite(m["false_alarm_rate"]) and m["false_alarm_rate"] <= false_alarm_budget + 1e-12:
            # При равном Caught предпочитаем меньше ложных тревог, затем более высокий порог.
            options.append((m["caught_rate"], -m["false_alarm_rate"], float(threshold), m))

    if not options:
        raise RuntimeError("Не найден порог в заданном бюджете ложных тревог.")
    options.sort(key=lambda item: (item[0], item[1], item[2]), reverse=True)
    best = options[0]
    return best[2], best[3]

# Вспомогательная таблица по разрядам — используется только для метрик, не для входов модели.
validation_shots = shot_info.loc[shot_info["shot_id"].isin(valid_ids)].copy()
validation_rows = df.loc[valid_mask, [
    "shot_id", "t", "t_tq_s", "disrupted_bool", "target_y"
]].copy().reset_index(drop=True)

results = []
fitted_models = {}
validation_scores = {}
feature_columns_used = {}

for variant, full_X in feature_sets.items():
    print(f"Обучение варианта: {variant} ...")
    X_train = full_X.loc[train_mask].copy()
    X_valid = full_X.loc[valid_mask].copy()
    y_train = y.loc[train_mask].copy()
    y_valid = y.loc[valid_mask].copy()

    # Убираем признаки, которые полностью пусты в train; те же колонки убираем из validation.
    usable_cols = X_train.columns[X_train.notna().any(axis=0)].tolist()
    if not usable_cols:
        raise ValueError(f"У варианта {variant} нет непустых признаков в train.")
    X_train = X_train[usable_cols]
    X_valid = X_valid[usable_cols]

    model = fit_classifier(X_train, y_train)
    p_valid = get_probabilities(model, X_valid)
    ranking = row_ranking_metrics(y_valid, p_valid)

    scored = validation_rows.copy()
    scored["p"] = p_valid
    threshold, operating = choose_threshold(
        scored, validation_shots, LEAD_TIME_S, FALSE_ALARM_BUDGET
    )

    result = {
        "variant": variant,
        "n_features": len(usable_cols),
        **ranking,
        **operating,
    }
    results.append(result)
    fitted_models[variant] = model
    validation_scores[variant] = p_valid
    feature_columns_used[variant] = usable_cols

results_df = pd.DataFrame(results).sort_values(
    ["caught_rate", "pr_auc", "false_alarm_rate"],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)

print("\nСРАВНЕНИЕ НА ОДИНАКОВОЙ VALIDATION-ВЫБОРКЕ")
display(results_df[[
    "variant", "n_features", "roc_auc", "pr_auc", "threshold",
    "caught", "n_events", "caught_rate", "late", "missed",
    "false_alarms", "n_safe", "false_alarm_rate"
]].style.format({
    "roc_auc": "{:.4f}", "pr_auc": "{:.4f}", "threshold": "{:.5f}",
    "caught_rate": "{:.1%}", "false_alarm_rate": "{:.1%}"
}))

Обучение варианта: Raw ...
Обучение варианта: Physics ...
Обучение варианта: Dynamics ...

СРАВНЕНИЕ НА ОДИНАКОВОЙ VALIDATION-ВЫБОРКЕ


,variant,n_features,roc_auc,pr_auc,threshold,caught,n_events,caught_rate,late,missed,false_alarms,n_safe,false_alarm_rate
0,Dynamics,197,0.9752,0.3769,0.52931,5,9,55.6%,1,3,0,15,0.0%
1,Physics,119,0.9756,0.4137,0.81824,4,9,44.4%,1,4,0,15,0.0%
2,Raw,101,0.9707,0.3888,0.86089,4,9,44.4%,2,3,0,15,0.0%


## 6. Выбор модели и сохранение

Первичный критерий — `caught_rate` при соблюдении бюджета ложных тревог; при равенстве используем PR-AUC, поскольку положительные строки редкие. Рейтинг на одном holdout не является окончательным доказательством: число разрядов может быть небольшим.

Сохраняем выбранный классификатор, обученный повторно на всех пригодных разрядах текущего набора, а также список признаков и метаданные. Файл `best_model.joblib` — **артефакт для экспериментов в Python**, а не готовая папка сдачи `toksym`: для запуска в симуляторе ещё нужно реализовать такой же причинный сбор признаков в `model.py` и проверить ограничения доступных библиотек.

In [7]:
best_variant = str(results_df.iloc[0]["variant"])
best_threshold_validation = float(results_df.iloc[0]["threshold"])
best_model = fitted_models[best_variant]
best_columns = feature_columns_used[best_variant]

print("Выбранный вариант:", best_variant)
print(f"Caught на validation при бюджете {FALSE_ALARM_BUDGET:.1%}: "
      f"{results_df.iloc[0]['caught_rate']:.1%}")
print(f"Ложные тревоги на validation: {results_df.iloc[0]['false_alarm_rate']:.1%}")
print(f"Validation threshold (диагностический): {best_threshold_validation:.6f}")

# Финальная локальная версия переобучается на всех разрядах текущего seed.
# Порог выше остаётся только диагностикой; в новом seed его нужно проверять заново.
X_all = feature_sets[best_variant][best_columns].copy()
y_all = y.copy()
final_model = fit_classifier(X_all, y_all)

bundle = {
    "model": final_model,
    "feature_variant": best_variant,
    "feature_columns": list(best_columns),
    "lead_time_s": float(LEAD_TIME_S),
    "false_alarm_budget": float(FALSE_ALARM_BUDGET),
    "validation_threshold_diagnostic_only": best_threshold_validation,
    "backend": MODEL_BACKEND,
    "random_state": RANDOM_STATE,
    "n_training_shots": int(shot_info["shot_id"].nunique()),
    "warning": (
        "Offline notebook artifact only. Recreate the same causal features in model.py "
        "before running in toksym; confirm on a different Master seed."
    ),
}

try:
    import joblib
    joblib.dump(bundle, "best_model.joblib")
    print("\nСохранено: best_model.joblib")
except ImportError:
    print("joblib не установлен — модель обучена в памяти, но не сохранена.")

results_df.to_csv("model_comparison.csv", index=False)
print("Сохранено: model_comparison.csv")

# Для деревьев печатаем самые важные признаки финальной модели, если estimator это поддерживает.
if hasattr(final_model, "feature_importances_"):
    importances = pd.Series(final_model.feature_importances_, index=best_columns).sort_values(ascending=False)
    print("\nТоп-25 признаков:")
    display(importances.head(25).to_frame("importance"))
else:
    print("У выбранного backend нет готового атрибута feature_importances_.")

Выбранный вариант: Dynamics
Caught на validation при бюджете 5.0%: 55.6%
Ложные тревоги на validation: 0.0%
Validation threshold (диагностический): 0.529313

Сохранено: best_model.joblib
Сохранено: model_comparison.csv

Топ-25 признаков:


,importance
bolo_05,0.216087
bolo_04,0.062119
ece_15,0.058510
coil_pf_radial_kA,0.058323
feat_q95_base_proxy,0.053778
ece_14,0.042472
bolo_03,0.034202
prog_p_nbi_MW,0.020537
prog_ip_MA,0.019925
coil_cs_flux_Vs,0.017212


## 7. Следующие шаги перед финальной сдачей

1. Повтори сбор данных с **другим `Master seed`**. Оцени выбранную модель на этих разрядах, не меняя её под каждую ошибку.
2. Проверь не только ROC-AUC/PR-AUC, но и `Caught`, `Late`, `Missed`, `False alarm` в `toksym Test run` при разных `Lead time` и False-alarm budget.
3. Перенеси ровно те же признаки в потоковую `Model.predict(t, channels)`. Лаги/окна должны обновляться только текущими и прошлыми каналами; состояние истории нужно сбрасывать в `start_shot`.
4. Не используй `t_tq_s`, `disrupted`, `scenario`, `spontaneous`, `wall_contact`, `t_end_s` или другие столбцы `shots.csv` в `X`. Они нужны только для меток и анализа.
5. Проверь скорость: `predict` вызывается каждые 5 ms, значит обработка одного шага должна быть лёгкой.

**Важно:** если доступно мало разрядов, разброс метрик будет большим. Добавление новых признаков имеет смысл только тогда, когда улучшение сохраняется на независимом seed.